In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import*

import os
import sys

project_pth=os.path.join(os.getcwd(),'..','..')
sys.path.append(project_pth)
from utils.transformations import reusable

In [0]:
project_pth

In [0]:
df_user=spark.readStream.format("cloudFiles").option("cloudFiles.format","parquet").option("cloudFiles.schemaLocation","abfss://silver@stoargeazureproject.dfs.core.windows.net/DimUser/checkpoint").option("schemaEvolution","addNewColumns").load("abfss://bronze@stoargeazureproject.dfs.core.windows.net/DimUser")


In [0]:
display(df_user)

In [0]:
from pyspark.sql.functions import col, upper
df_user=df_user.withColumn("user_name",upper(col("user_name")))
display(df_user)             

In [0]:
import os
import sys

project_pth=os.path.join(os.getcwd(),os.path.pardir)
sys.path.append(project_pth)
from utils.transformations import *

In [0]:
df_user_obj=reusable()
df_user= df_user_obj.dropColumns(df_user,['_rescued_data'])
df_user= df_user.dropDuplicates(['user_id'])
display(df_user)

In [0]:
df_user.writeStream.format("delta").outputMode("append").option("checkpointLocation","abfss://silver@stoargeazureproject.dfs.core.windows.net/DimUser/checkpoint").trigger(availableNow=True).option("path","abfss://silver@stoargeazureproject.dfs.core.windows.net/DimUser/data").toTable("spotify_cata.silver.DimUser")

In [0]:
## Artist data

df_art=spark.readStream.format("cloudFiles").option("cloudFiles.format","parquet").option("cloudFiles.schemaLocation","abfss://silver@stoargeazureproject.dfs.core.windows.net/DimUser/checkpoint").option("schemaEvolution","addNewColumns").load("abfss://bronze@stoargeazureproject.dfs.core.windows.net/DimArtist")

In [0]:
display(df_art)

In [0]:
df_art_obj=reusable()
df_art= df_art_obj.dropColumns(df_art,['_rescued_data'])
df_art= df_art.dropDuplicates(['artist_id'])
display(df_art)


In [0]:
df_art.writeStream.format("delta").outputMode("append").option("checkpointLocation","abfss://silver@stoargeazureproject.dfs.core.windows.net/DimArt/checkpoint").trigger(once=True).option("path","abfss://silver@stoargeazureproject.dfs.core.windows.net/DimArt/data").toTable("spotify_cata.silver.DimArtist")

In [0]:
df_Track=spark.readStream.format("cloudFiles").option("cloudFiles.format","parquet").option("cloudFiles.schemaLocation","abfss://silver@stoargeazureproject.dfs.core.windows.net/DimTrack/checkpoint").option("schemaEvolution","addNewColumns").load("abfss://bronze@stoargeazureproject.dfs.core.windows.net/DimTrack")

In [0]:
display(df_Track)


In [0]:
df_Track= df_Track.withColumn('durationFlag',when(col('duration_sec')<150,'low')\
                              .when(col('duration_sec')<300,'medium')\
                              .otherwise('high'))

df_Track=df_Track.withColumn("track_name",regexp_replace(col("track_name"),'-',' '))
df_Track= reusable().dropColumns(df_Track,['_rescued_data'])
display(df_Track)

In [0]:
df_Track.writeStream.format("delta").outputMode("append").option("checkpointLocation","abfss://silver@stoargeazureproject.dfs.core.windows.net/DimTrack/checkpoint").trigger(once=True).option("path","abfss://silver@stoargeazureproject.dfs.core.windows.net/DimTrack/data").toTable("spotify_cata.silver.DimTrack")

In [0]:
df_dt=spark.readStream.format("cloudFiles").option("cloudFiles.format","parquet").option("cloudFiles.schemaLocation","abfss://silver@stoargeazureproject.dfs.core.windows.net/DimDate/checkpoint").option("schemaEvolution","addNewColumns").load("abfss://bronze@stoargeazureproject.dfs.core.windows.net/DimDate")

In [0]:
display(df_dt)

In [0]:
df_dt= reusable().dropColumns(df_dt,*['_rescued_data'])
display(df_dt)


In [0]:
df_dt.writeStream.format("delta").outputMode("append").option("checkpointLocation","abfss://silver@stoargeazureproject.dfs.core.windows.net/DimDate/checkpoint").trigger(once=True).option("path","abfss://silver@stoargeazureproject.dfs.core.windows.net/DimDate/data").toTable("spotify_cata.silver.DimDate")

In [0]:
df_ft=spark.readStream.format("cloudFiles").option("cloudFiles.format","parquet").option("cloudFiles.schemaLocation","abfss://silver@stoargeazureproject.dfs.core.windows.net/FactStream/checkpoint").option("schemaEvolution","addNewColumns").load("abfss://bronze@stoargeazureproject.dfs.core.windows.net/FactStream")

In [0]:
display(df_ft)

In [0]:
df_ft= reusable().dropColumns(df_ft,['_rescued_data'])
display(df_ft)


In [0]:
df_ft.writeStream.format("delta").outputMode("append").option("checkpointLocation","abfss://silver@stoargeazureproject.dfs.core.windows.net/FactStream/checkpoint").trigger(once=True).option("path","abfss://silver@stoargeazureproject.dfs.core.windows.net/FactStream/data").toTable("spotify_cata.silver.FactStream")